In [1]:
import argparse
import itertools
import os
import pathlib
import sys
from functools import reduce

import duckdb
import pandas as pd
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm
profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir  # default to root_dir instead of NAS

In [2]:
patient_id_file = pathlib.Path(f"{profile_base_dir}/data/patient_IDs.txt").resolve(
    strict=True
)
patients = pd.read_csv(
    patient_id_file, header=None, names=["patient_id"]
).patient_id.tolist()

In [3]:
out_dict = {
    "file_path": [],
    "patient_id": [],
    "well_fov": [],
    "feature_type": [],
    "compartment": [],
    # "df_shape": [],
}

# get all well_fovs for a patient
for patient in tqdm.tqdm(patients, desc="Processing patients", leave=True):
    patient_dir = profile_base_dir / "data" / patient / "extracted_features"
    well_fovs = patient_dir.glob("*")  # get all well_fovs for a patient
    # print(f"Found well_fovs: {well_fovs}")
    for well_fov in tqdm.tqdm(well_fovs, desc="Processing well_fovs", leave=False):
        if "stats" in well_fov.stem:
            continue
        features = pathlib.Path(well_fov).glob("*.parquet")
        for feature in features:
            feature_type = feature.stem.split("_")[2]
            compartment = feature.stem.split("_")[0]
            out_dict["file_path"].append(feature)
            out_dict["patient_id"].append(patient)
            out_dict["well_fov"].append(feature.parent.stem)
            out_dict["feature_type"].append(feature_type)
            out_dict["compartment"].append(compartment)
            # out_dict["df_shape"].append(pd.read_parquet(feature).shape)
df = pd.DataFrame(out_dict)
# df = df.loc[df["patient_id"] == "NF0014_T1"]
df = df.loc[(df["feature_type"] == "Granularity") & (df["compartment"] != "Organoid")]
df = df.loc[(df["compartment"] != "Organoid")]

df.head()

Processing patients:   0%|          | 0/13 [00:00<?, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

...

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

,file_path,patient_id,well_fov,feature_type,compartment
1,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell
5,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cytoplasm
16,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cytoplasm
23,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell
27,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell


In [4]:
from tqdm import tqdm

tqdm.pandas()


def safe_read_shape(x):
    try:
        df = pd.read_parquet(x)
        return df.shape, df.isna().sum().sum()
    except Exception as e:
        print(f"Error reading {x}: {e}")
        return None, None


# if not pathlib.Path("../logs/feature_file_info.parquet").exists():
df[["df_shape", "missing_values"]] = df["file_path"].progress_apply(
    lambda x: pd.Series(safe_read_shape(x))
)
df["file_path"] = df["file_path"].astype(str)
df.to_parquet("../logs/feature_file_info.parquet", index=False)
# else:
#     df = pd.read_parquet("../logs/feature_file_info.parquet")

100%|██████████| 34517/34517 [01:04<00:00, 533.24it/s]


In [5]:
df.sort_values(["patient_id", "well_fov"], inplace=True)
df.reset_index(drop=True, inplace=True)
df

,file_path,patient_id,well_fov,feature_type,compartment,df_shape,missing_values
0,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cell,"(13, 18)",0
1,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cytoplasm,"(13, 18)",0
2,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cytoplasm,"(13, 18)",0
3,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cell,"(13, 18)",0
4,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cell,"(13, 18)",0
...,...,...,...,...,...,...,...
34512,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-6,Granularity,Cell,"(13, 18)",0
34513,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Cytoplasm,"(18, 18)",0
34514,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Cytoplasm,"(18, 18)",0
34515,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Cytoplasm,"(18, 18)",0


In [6]:
# merge the cells, cytoplasm, and whole cell features for a given well_fov and patient_id
# check for missing values and shape of the dataframes
out_dict = {
    "patient_id": [],
    "well_fov": [],
    "path": [],
    "type": [],
    "feature_type": [],
}
for row in tqdm(
    df.itertuples(), total=df.shape[0], desc="Merging features", leave=True
):
    out_dict["patient_id"].append(row.patient_id)
    out_dict["well_fov"].append(row.well_fov)
    out_dict["path"].append(row.file_path)
    out_dict["type"].append(f"{row.compartment}")
    out_dict["feature_type"].append(row.feature_type)
out_df = pd.DataFrame(out_dict)
out_df.drop_duplicates(subset=["patient_id", "well_fov", "type"], inplace=True)
# pivot such that each type has its own column
out_df = out_df.pivot(
    index=[
        "patient_id",
        "well_fov",
    ],
    columns="type",
    values="path",
).reset_index()

Merging features: 100%|██████████| 34517/34517 [00:00<00:00, 709711.66it/s]


In [7]:
labels_dict = {
    "Cell_labels": [],
    "Cytoplasm_labels": [],
    "Nuclei_labels": [],
    "patient_id": [],
    "well_fov": [],
}

# merge the dataframes and check for missing values and shape
for row in tqdm(
    out_df.itertuples(),
    total=out_df.shape[0],
    desc="Checking merged features",
    leave=True,
):
    try:
        cell_df = pd.read_parquet(row.Cell)
        cytoplasm_df = pd.read_parquet(row.Cytoplasm)
        nuclei_df = pd.read_parquet(row.Nuclei)
        labels_dict["Cell_labels"].append(cell_df["object_id"].tolist())
        labels_dict["Cytoplasm_labels"].append(cytoplasm_df["object_id"].tolist())
        labels_dict["Nuclei_labels"].append(nuclei_df["object_id"].tolist())
        labels_dict["patient_id"].append(row.patient_id)
        labels_dict["well_fov"].append(row.well_fov)
    except Exception as e:
        print(f"Error reading files for {row.patient_id} {row.well_fov}: {e}")
labels_df = pd.DataFrame(labels_dict)

Checking merged features:  15%|█▍        | 481/3317 [00:02<00:11, 241.78it/s]

Error reading files for NF0016_T1 D3-1: cannot construct a FileSource from nan
Error reading files for NF0016_T1 D6-1: cannot construct a FileSource from nan
Error reading files for NF0016_T1 D8-1: cannot construct a FileSource from nan


Checking merged features:  34%|███▍      | 1137/3317 [00:04<00:04, 487.70it/s]

Error reading files for NF0030_T1 C10-1: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C10-2: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C10-3: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C11-2: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C11-4: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C2-1: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C2-2: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C2-3: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C2-4: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C3-1: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C3-2: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C3-3: cannot construct a FileSource from nan
Error reading files for NF0030_T1 C4-1: cannot 

Checking merged features:  37%|███▋      | 1219/3317 [00:04<00:03, 582.35it/s]

...

Error reading files for NF0055_T1 F3-2: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C10-3: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C10-4: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C11-1: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C11-2: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C11-4: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C2-1: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C2-3: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C3-1: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C3-4: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C4-3: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C4-4: cannot construct a FileSource from nan
Error reading files for S

Checking merged features:  95%|█████████▍| 3150/3317 [00:11<00:00, 698.92it/s]

Error reading files for SARCO361_T1 C10-4: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C10-6: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C10-7: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C11-2: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C11-3: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C11-4: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C11-5: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C11-7: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C2-1: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C2-3: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C2-5: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 C2-6: cannot construct a FileSource from nan
Error reading files 

Checking merged features: 100%|██████████| 3317/3317 [00:11<00:00, 289.87it/s]

Error reading files for SARCO361_T1 E7-3: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E7-6: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E8-3: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E8-4: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E8-5: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E8-7: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E9-1: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E9-2: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E9-3: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E9-4: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E9-5: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E9-7: cannot construct a FileSource from nan
Error reading files for SARC

In [8]:
labels_df["labels_match"] = labels_df.apply(
    lambda row: row["Cell_labels"] == row["Nuclei_labels"],
    axis=1,
)
labels_df["same_number_of_labels"] = labels_df.apply(
    lambda row: len(row["Cell_labels"]) == len(row["Nuclei_labels"]),
    axis=1,
)
labels_df["unique_labels_across_compartments"] = labels_df.apply(
    lambda row: set(row["Nuclei_labels"]) - set(row["Cytoplasm_labels"]), axis=1
)
labels_df.loc[labels_df["labels_match"] == False]

,Cell_labels,Cytoplasm_labels,Nuclei_labels,patient_id,well_fov,labels_match,same_number_of_labels,unique_labels_across_compartments
103,"[1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15, 1...","[1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15, 1...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0014_T2,C10-1,False,False,"{32, 65, 69, 40, 9, 12, 77, 46, 55, 22, 54, 24..."
331,"[2, 3, 4, 5, 6, 7]","[2, 3, 4, 5, 6, 7, 8]","[1, 2, 3, 4, 5, 6, 7, 8]",NF0014_T2,F5-4,False,False,{1}
383,"[1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15]","[1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 1...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0014_T2,G3-5,False,False,"{4, 14}"
580,"[1, 3, 4, 5, 6, 8, 11, 12, 14, 15, 16, 17, 18,...","[1, 3, 4, 5, 6, 8, 11, 12, 14, 15, 16, 17, 18,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0018_T6,D4-2,False,False,"{33, 2, 7, 9, 10, 13, 19, 51, 24, 58}"
758,"[1, 2, 4, 5, 7, 9, 10, 11, 12, 13, 14, 16, 17,...","[1, 2, 4, 5, 7, 9, 10, 11, 12, 13, 14, 16, 17,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",NF0021_T1,D11-1,False,False,"{8, 3, 6, 15}"
...,...,...,...,...,...,...,...,...
2543,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",SARCO219_T2,E9-4,False,False,"{131, 141, 142, 15, 145, 149, 24, 153, 33, 163..."
2551,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, ...","[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",SARCO219_T2,F7-4,False,False,"{1, 2, 138, 139, 15, 17, 149, 24, 156, 32, 34,..."
2555,"[1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 1...","[1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 1...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",SARCO219_T2,F9-3,False,False,"{131, 8, 15, 45, 46, 53, 57, 59, 65, 68, 74, 7..."
2564,"[2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...","[2, 3, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",SARCO219_T2,G9-1,False,False,"{1, 4, 5, 133, 134, 144, 20, 153, 154, 29, 33,..."


In [9]:
# get the patient_id and well_fov for the rows where the labels do not match and check the corresponding feature files for those rows
mismatched_labels = labels_df.loc[
    labels_df["labels_match"] == False, ["patient_id", "well_fov"]
]
mismatched_labels

,patient_id,well_fov
103,NF0014_T2,C10-1
331,NF0014_T2,F5-4
383,NF0014_T2,G3-5
580,NF0018_T6,D4-2
758,NF0021_T1,D11-1
...,...,...
2543,SARCO219_T2,E9-4
2551,SARCO219_T2,F7-4
2555,SARCO219_T2,F9-3
2564,SARCO219_T2,G9-1


In [10]:
labels_df["patient_id"].unique()

array(['NF0014_T1', 'NF0014_T2', 'NF0016_T1', 'NF0018_T6', 'NF0021_T1',
       'NF0030_T1', 'NF0035_T1', 'NF0037_T1', 'NF0037_T1_CQ1',
       'NF0040_T1', 'SARCO219_T2', 'SARCO361_T1'], dtype=object)

In [11]:
labels_df.loc[labels_df["same_number_of_labels"] == False].value_counts("patient_id")

patient_id
NF0037_T1      44
NF0040_T1      25
SARCO219_T2     6
NF0021_T1       4
NF0014_T2       3
NF0035_T1       3
NF0030_T1       2
NF0018_T6       1
SARCO361_T1     1
Name: count, dtype: int64

In [12]:
# # show the well fov and the patient id
# # print the unique well fovs that have mismatched labels
# for row in labels_df.loc[labels_df["labels_match"] == False][
#     ["patient_id", "well_fov"]
# ].itertuples():
#     print(f"cd ../../{row.patient_id}/extracted_features/ ; rm -r {row.well_fov}")

In [13]:
tmp_df = pd.merge(
    left=pd.merge(
        left=cell_df,
        right=cytoplasm_df,
        on=["object_id", "image_set"],
    ),
    right=nuclei_df,
    on=["object_id", "image_set"],
)
tmp_df.head()

,image_set,object_id,Cell_AGP_Granularity_1,Cell_AGP_Granularity_2,Cell_AGP_Granularity_3,Cell_AGP_Granularity_4,Cell_AGP_Granularity_5,Cell_AGP_Granularity_6,Cell_AGP_Granularity_7,Cell_AGP_Granularity_8,...,Nuclei_Mito_Granularity_7,Nuclei_Mito_Granularity_8,Nuclei_Mito_Granularity_9,Nuclei_Mito_Granularity_10,Nuclei_Mito_Granularity_11,Nuclei_Mito_Granularity_12,Nuclei_Mito_Granularity_13,Nuclei_Mito_Granularity_14,Nuclei_Mito_Granularity_15,Nuclei_Mito_Granularity_16
0,G9-6,1,16.104172,1.970163,1.529468,6.219732,5.100858,6.722161,8.139821,0.0,...,3.936938,3.967296,3.975250,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
1,G9-6,2,39.611074,14.116968,0.000000,12.159025,6.784916,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
2,G9-6,3,9.137148,0.145719,0.598893,6.369452,7.206556,0.000000,0.000000,0.0,...,0.000000,0.000000,3.507180,0.0,0.0,0.0,3.533530,0.0,0.0,3.564691
3,G9-6,4,2.067450,2.589675,2.903647,10.183098,8.004644,0.000000,0.000000,0.0,...,0.000000,0.000000,3.944703,0.0,0.0,0.0,4.120513,0.0,0.0,4.252967
4,G9-6,5,2.767507,1.335198,1.669861,7.917151,8.020715,0.000000,0.000000,0.0,...,0.000000,0.000000,3.739917,0.0,0.0,0.0,3.766272,0.0,0.0,3.772596
